# Laboratorium 12: Kalibracja kamery (Model Pinhole)

**Cel:** Zrozumienie procesu kalibracji kamery, wyznaczanie parametrów wewnętrznych i zewnętrznych oraz korekcja zniekształceń optycznych.

---

## Sekcja 1: Teoria

### 1.1 Model kamery Pinhole (otworkowej)

**Model pinhole** to matematyczny model kamery, który opisuje jak punkty 3D w świecie rzeczywistym są rzutowane na płaszczyznę obrazu 2D.

**Zasada działania:**
```
Świat 3D          Otwór (pinhole)      Płaszczyzna obrazu 2D
                                       
    P(X,Y,Z) ---------> • -----------> p(x,y)
                      otwór
                    (centrum)
```

**Intuicja Feynmana:**
> Wyobraź sobie pudełko z małym otworkiem. Światło z punktu w przestrzeni 3D przechodzi przez otwór i tworzy obraz na tylnej ścianie pudełka. To jest właśnie model pinhole!

**Równanie projekcji perspektywicznej:**

Punkt 3D: **P = (X, Y, Z)** w układzie kamery  
Punkt 2D: **p = (x, y)** na płaszczyźnie obrazu

$$
x = f \cdot \frac{X}{Z}, \quad y = f \cdot \frac{Y}{Z}
$$

gdzie **f** to ogniskowa kamery.

**Wizualizacja:**
```
        Z (oś optyczna)
        ↑
        |
    P(X,Y,Z)
       /|\
      / | \
     /  |  \
    /   |   \
   /    •----→ X
  /  centrum
 /   kamery
|----------|
  płaszczyzna
    obrazu
```

---

### 1.2 Parametry wewnętrzne kamery (Intrinsic Parameters)

**Macierz kamery K** (3×3) opisuje wewnętrzne właściwości kamery:

$$
K = \begin{bmatrix}
f_x & 0 & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$

**Parametry:**
- **f_x, f_y** – ogniskowa w pikselach (oś X i Y)
- **c_x, c_y** – punkt główny (principal point) – środek obrazu w pikselach

**Dlaczego f_x ≠ f_y?**
- Piksele mogą nie być kwadratowe (aspect ratio)
- Zniekształcenia optyczne

**Punkt główny (c_x, c_y):**
- Punkt przecięcia osi optycznej z płaszczyzną obrazu
- Zazwyczaj blisko środka obrazu, ale nie zawsze dokładnie w środku

---

### 1.3 Parametry zewnętrzne (Extrinsic Parameters)

**Parametry zewnętrzne** opisują pozycję i orientację kamery w przestrzeni 3D.

**Transformacja z układu świata do układu kamery:**

$$
\begin{bmatrix} X_c \\ Y_c \\ Z_c \end{bmatrix} = R \begin{bmatrix} X_w \\ Y_w \\ Z_w \end{bmatrix} + t
$$

gdzie:
- **R** – macierz rotacji (3×3)
- **t** – wektor translacji (3×1)
- **(X_w, Y_w, Z_w)** – współrzędne w układzie świata
- **(X_c, Y_c, Z_c)** – współrzędne w układzie kamery

**Wektor rotacji (rvec):**
- Reprezentacja Rodriguesa: wektor 3D, którego kierunek określa oś rotacji, a długość kąt obrotu
- Konwersja do macierzy R: `cv2.Rodrigues(rvec)`

---

### 1.4 Zniekształcenia optyczne (Lens Distortion)

**Problem:** Rzeczywiste soczewki nie są idealne – powodują zniekształcenia obrazu.

**Typy zniekształceń:**

#### A) Zniekształcenia radialne (Radial Distortion)
- Najczęstszy typ zniekształceń
- Punkty są przesuwane wzdłuż promieni od centrum obrazu

**Współczynniki:** k₁, k₂, k₃

$$
x_{distorted} = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)
$$
$$
y_{distorted} = y(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)
$$

gdzie: $r^2 = x^2 + y^2$ (odległość od centrum)

**Typy:**
- **Barrel distortion** (k₁ < 0): linie proste wyginają się na zewnątrz
- **Pincushion distortion** (k₁ > 0): linie proste wyginają się do środka

#### B) Zniekształcenia tangencjalne (Tangential Distortion)
- Występują gdy soczewka nie jest równoległa do płaszczyzny obrazu

**Współczynniki:** p₁, p₂

**Wektor współczynników zniekształceń:**
```
dist_coeffs = [k₁, k₂, p₁, p₂, k₃]
```

---

### 1.5 Proces kalibracji kamery

**Cel:** Wyznaczyć parametry wewnętrzne (K, dist_coeffs) i zewnętrzne (R, t) kamery.

**Wzorzec kalibracyjny – szachownica:**
- Płaski wzorzec z czarnymi i białymi kwadratami
- Znane wymiary kwadratów (np. 25mm × 25mm)
- Łatwe do wykrycia narożniki wewnętrzne

**Algorytm kalibracji:**

1. **Przygotowanie danych:**
   - Zrób 10-20 zdjęć szachownicy z różnych kątów
   - Wykryj narożniki na każdym obrazie
   - Przygotuj punkty 3D wzorca (znane współrzędne)

2. **Wykrywanie narożników:**
   - `cv2.findChessboardCorners()` – wykrywa narożniki
   - `cv2.cornerSubPix()` – precyzja subpikselowa

3. **Kalibracja:**
   - `cv2.calibrateCamera()` – wyznacza K, dist_coeffs, R, t
   - Minimalizuje błąd reprojection

4. **Korekcja zniekształceń:**
   - `cv2.undistort()` – usuwa zniekształcenia z obrazu

**Błąd reprojection (RMS):**
- Miara jakości kalibracji
- **Dobra kalibracja:** RMS < 0.5 piksela
- **Akceptowalna:** RMS < 1.0 piksela

---

## Sekcja 2: Kod startowy

### 2.1 Import bibliotek

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import patches
import glob
import os

# Konfiguracja wyświetlania
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['image.cmap'] = 'gray'

### 2.2 Funkcje pomocnicze

In [ ]:
def show_images(images, titles, cmap='gray', figsize=(15, 5)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap=cmap)
        axes[i].set_title(title, fontsize=12)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def print_camera_params(camera_matrix, dist_coeffs):
    print("=" * 50)
    print("PARAMETRY KAMERY")
    print("=" * 50)
    print("Macierz kamery K:")
    print(camera_matrix)
    print(f"Ogniskowa: fx={camera_matrix[0,0]:.2f}, fy={camera_matrix[1,1]:.2f}")
    print(f"Punkt główny: cx={camera_matrix[0,2]:.2f}, cy={camera_matrix[1,2]:.2f}")
    print(f"Zniekształcenia: k1={dist_coeffs[0][0]:.6f}, k2={dist_coeffs[0][1]:.6f}")
    print("=" * 50)

### 2.3 Pobieranie obrazów kalibracyjnych z OpenCV

In [ ]:
def download_calibration_images(num_images=16, save_dir='calibration_images'):
    import urllib.request
    
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    base_url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/left"
    image_paths = []
    print(f"Pobieranie {num_images} obrazów...")
    
    for i in range(1, num_images + 1):
        filename = f"left{i:02d}.jpg"
        url = f"{base_url}{i:02d}.jpg"
        save_path = os.path.join(save_dir, filename)
        
        if not os.path.exists(save_path):
            try:
                urllib.request.urlretrieve(url, save_path)
                print(f"  ✓ {filename}")
            except:
                continue
        image_paths.append(save_path)
    
    print(f"Pobrano {len(image_paths)} obrazów")
    return image_paths

def load_calibration_images(image_dir='calibration_images'):
    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
    
    if len(image_paths) == 0:
        image_paths = download_calibration_images()
    
    images = []
    for path in image_paths:
        img = cv2.imread(path)
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            images.append(gray)
    
    print(f"Załadowano {len(images)} obrazów")
    return images, image_paths

### 2.4 Wczytanie obrazów

In [ ]:
calibration_images, image_paths = load_calibration_images()

if len(calibration_images) >= 4:
    show_images(calibration_images[:4], [f'Obraz {i+1}' for i in range(4)], figsize=(20, 5))
    print(f"Rozmiar: {calibration_images[0].shape}")

### 2.5 Parametry wzorca

In [ ]:
PATTERN_SIZE = (9, 6)
SQUARE_SIZE = 1.0

print(f"Wzorzec: {PATTERN_SIZE[0]} × {PATTERN_SIZE[1]} narożników")
print(f"Liczba punktów: {PATTERN_SIZE[0] * PATTERN_SIZE[1]}")

---

## Koniec Części 1

W następnej części zaimplementujemy zadania kalibracyjne.

---

---
# **ZADANIE 1: Przygotowanie danych kalibracyjnych**

## **Cel zadania:**
Wykrycie narożników szachownicy na obrazach kalibracyjnych i przygotowanie par punktów: współrzędne 3D w układzie świata rzeczywistego oraz odpowiadające im współrzędne 2D na obrazie.

## **Teoria:**

### **Wzorzec kalibracyjny - szachownica**
Szachownica jest najpopularniejszym wzorcem kalibracyjnym ze względu na:
- Łatwość wykrywania narożników z subpikselową dokładnością
- Znane wymiary geometryczne (rozmiar kwadratu)
- Regularna struktura ułatwiająca automatyczne wykrywanie

### **Układ współrzędnych wzorca**
Definiujemy układ współrzędnych 3D na płaszczyźnie szachownicy:
- Początek układu: lewy górny narożnik wewnętrzny
- Oś X: w prawo wzdłuż pierwszego wiersza
- Oś Y: w dół wzdłuż pierwszej kolumny  
- Oś Z: prostopadle do płaszczyzny szachownicy (Z = 0 dla wszystkich punktów)

Dla szachownicy o wymiarach wewnętrznych (nx, ny) i rozmiarze kwadratu `square_size`:
```
Punkt (i, j): [j * square_size, i * square_size, 0]
```

### **Funkcje OpenCV:**

#### `cv2.findChessboardCorners(image, patternSize, flags)`
Wykrywa narożniki szachownicy na obrazie.
- **image**: Obraz w skali szarości
- **patternSize**: Tuple (nx, ny) - liczba narożników wewnętrznych
- **flags**: Flagi wykrywania (np. `cv2.CALIB_CB_ADAPTIVE_THRESH`)
- **Zwraca**: (ret, corners) - status i współrzędne narożników

#### `cv2.cornerSubPix(image, corners, winSize, zeroZone, criteria)`
Rafinuje pozycje narożników z dokładnością subpikselową.
- **winSize**: Rozmiar okna wyszukiwania (np. (11, 11))
- **zeroZone**: Martwa strefa (-1, -1 = brak)
- **criteria**: Kryteria zakończenia algorytmu

#### `cv2.drawChessboardCorners(image, patternSize, corners, patternWasFound)`
Rysuje wykryte narożniki na obrazie do wizualizacji.

## **Implementacja:**


In [ ]:
# ZADANIE 1.1: Funkcja generująca współrzędne 3D punktów wzorca

def generate_object_points(pattern_size, square_size):
    """
    Generuje współrzędne 3D punktów wzorca (szachownicy) w układzie świata rzeczywistego.

    Parametry:
    ----------
    pattern_size : tuple (nx, ny)
        Liczba narożników wewnętrznych szachownicy (kolumny, wiersze)
    square_size : float
        Rozmiar pojedynczego kwadratu szachownicy w jednostkach rzeczywistych (np. mm)

    Zwraca:
    -------
    object_points : ndarray, shape (nx*ny, 3)
        Tablica współrzędnych 3D punktów wzorca, gdzie Z=0 dla wszystkich punktów
    """
    # TODO: Zaimplementuj generowanie współrzędnych 3D
    # Wskazówka: Użyj np.mgrid lub pętli do wygenerowania siatki punktów
    # Pamiętaj, że Z=0 dla wszystkich punktów (płaska szachownica)

    pass

# Test funkcji
pattern_size = (9, 6)
square_size = 25.0  # mm

object_points = generate_object_points(pattern_size, square_size)
print(f"Wygenerowano {len(object_points)} punktów 3D")
print(f"Pierwsze 5 punktów:\n{object_points[:5]}")


---
## **Kontynuacja ZADANIA 1**


In [ ]:
# ZADANIE 1.2: Funkcja wykrywająca narożniki szachownicy

def detect_chessboard_corners(image, pattern_size):
    """
    Wykrywa narożniki szachownicy na obrazie z subpikselową dokładnością.

    Parametry:
    ----------
    image : ndarray
        Obraz wejściowy (BGR lub skala szarości)
    pattern_size : tuple (nx, ny)
        Liczba narożników wewnętrznych szachownicy (kolumny, wiersze)

    Zwraca:
    -------
    success : bool
        True jeśli wykryto wszystkie narożniki, False w przeciwnym razie
    corners : ndarray, shape (nx*ny, 1, 2) lub None
        Współrzędne 2D wykrytych narożników z subpikselową dokładnością
    """
    # TODO: Zaimplementuj wykrywanie narożników
    # 1. Konwersja do skali szarości (jeśli obraz kolorowy)
    # 2. Użyj cv2.findChessboardCorners() do wykrycia narożników
    #    Flagi: cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE
    # 3. Jeśli wykryto, użyj cv2.cornerSubPix() do rafinacji pozycji
    #    Kryteria: (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    #    Rozmiar okna: (11, 11)
    # 4. Zwróć status i współrzędne narożników

    pass

# Test funkcji
test_image = calibration_images[0]
success, corners = detect_chessboard_corners(test_image, pattern_size)

if success:
    print(f"✓ Wykryto {len(corners)} narożników")

    # Wizualizacja
    img_with_corners = test_image.copy()
    cv2.drawChessboardCorners(img_with_corners, pattern_size, corners, success)

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(img_with_corners, cv2.COLOR_BGR2RGB))
    plt.title('Wykryte narożniki szachownicy')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("✗ Nie udało się wykryć narożników")


---
## **Kontynuacja ZADANIA 1**


In [ ]:
# ZADANIE 1.3: Przetworzenie wszystkich obrazów kalibracyjnych

def process_calibration_images(images, pattern_size, square_size):
    """
    Przetwarza wszystkie obrazy kalibracyjne i zbiera pary punktów 3D-2D.

    Parametry:
    ----------
    images : list
        Lista obrazów kalibracyjnych
    pattern_size : tuple (nx, ny)
        Liczba narożników wewnętrznych szachownicy
    square_size : float
        Rozmiar kwadratu szachownicy w jednostkach rzeczywistych

    Zwraca:
    -------
    object_points_list : list
        Lista tablic współrzędnych 3D dla każdego udanego obrazu
    image_points_list : list
        Lista tablic współrzędnych 2D dla każdego udanego obrazu
    valid_images : list
        Lista indeksów obrazów, na których wykryto szachownicę
    image_size : tuple (width, height)
        Rozmiar obrazów (potrzebny do kalibracji)
    """
    # TODO: Zaimplementuj przetwarzanie wszystkich obrazów
    # 1. Wygeneruj współrzędne 3D wzorca (te same dla wszystkich obrazów)
    # 2. Dla każdego obrazu wykryj narożniki
    # 3. Zbierz pary punktów 3D-2D dla udanych wykryć
    # 4. Zapisz rozmiar obrazu (width, height)
    # 5. Zwróć listy punktów i informacje o przetwarzaniu

    pass

# Wykonanie przetwarzania
object_points_list, image_points_list, valid_images, image_size = process_calibration_images(
    calibration_images, 
    pattern_size, 
    square_size
)

print(f"\nRozmiar obrazów: {image_size}")
print(f"Liczba zebranych par punktów: {len(object_points_list)}")


In [ ]:
# Wizualizacja wykrytych narożników na wszystkich udanych obrazach

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.ravel()

for i, img_idx in enumerate(valid_images):
    if i >= 16:
        break

    img = calibration_images[img_idx].copy()
    corners = image_points_list[i]

    cv2.drawChessboardCorners(img, pattern_size, corners, True)

    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Obraz {img_idx+1}', fontweight='bold')
    axes[i].axis('off')

for i in range(len(valid_images), 16):
    axes[i].axis('off')

plt.suptitle('Wykryte narożniki na wszystkich obrazach kalibracyjnych', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


---
# **ZADANIE 2: Kalibracja kamery**

## **Cel zadania:**
Obliczenie parametrów wewnętrznych kamery (macierz kamery K) oraz współczynników dystorsji na podstawie zebranych par punktów 3D-2D.

## **Teoria:**

### **Problem kalibracji**
Kalibracja kamery polega na znalezieniu parametrów, które minimalizują błąd reprojekcji:

$$
\min_{K, D, R_i, t_i} \sum_{i=1}^{N} \sum_{j=1}^{M} \| p_{ij} - \pi(K, D, R_i, t_i, P_j) \|^2
$$

gdzie:
- $p_{ij}$ - wykryty punkt 2D na obrazie $i$
- $P_j$ - punkt 3D wzorca
- $\pi$ - funkcja projekcji z uwzględnieniem dystorsji
- $K$ - macierz kamery
- $D$ - współczynniki dystorsji
- $R_i, t_i$ - rotacja i translacja dla obrazu $i$

### **Macierz kamery K**
Macierz parametrów wewnętrznych:

$$
K = \begin{bmatrix}
f_x & 0 & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$

gdzie:
- $f_x, f_y$ - ogniskowa w pikselach (oś X i Y)
- $c_x, c_y$ - współrzędne punktu głównego (optical center)

### **Model dystorsji**
Współczynniki dystorsji radialnej i tangencjalnej:

$$
D = [k_1, k_2, p_1, p_2, k_3]
$$

**Dystorsja radialna** (od środka obrazu):
$$
\begin{align}
x_{\text{distorted}} &= x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) \\
y_{\text{distorted}} &= y(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)
\end{align}
$$

**Dystorsja tangencjalna** (nierównoległość soczewki i sensora):
$$
\begin{align}
x_{\text{distorted}} &= x + [2p_1 xy + p_2(r^2 + 2x^2)] \\
y_{\text{distorted}} &= y + [p_1(r^2 + 2y^2) + 2p_2 xy]
\end{align}
$$

### **Funkcje OpenCV:**

#### `cv2.calibrateCamera(objectPoints, imagePoints, imageSize, cameraMatrix, distCoeffs, flags)`
Wykonuje kalibrację kamery.
- **objectPoints**: Lista tablic punktów 3D dla każdego obrazu
- **imagePoints**: Lista tablic punktów 2D dla każdego obrazu
- **imageSize**: Rozmiar obrazów (width, height)
- **cameraMatrix**: Początkowa macierz kamery (None = auto)
- **distCoeffs**: Początkowe współczynniki dystorsji (None = auto)
- **Zwraca**: (ret, cameraMatrix, distCoeffs, rvecs, tvecs)
  - ret: RMS błędu reprojekcji
  - rvecs: Lista wektorów rotacji dla każdego obrazu
  - tvecs: Lista wektorów translacji dla każdego obrazu

#### `cv2.projectPoints(objectPoints, rvec, tvec, cameraMatrix, distCoeffs)`
Projektuje punkty 3D na płaszczyznę obrazu 2D.
- Używane do obliczania błędu reprojekcji
- **Zwraca**: (imagePoints, jacobian)

## **Implementacja:**


In [ ]:
# ZADANIE 2.1: Funkcja kalibracji kamery

def calibrate_camera(object_points, image_points, image_size):
    """
    Wykonuje kalibrację kamery na podstawie par punktów 3D-2D.

    Parametry:
    ----------
    object_points : list
        Lista tablic współrzędnych 3D punktów wzorca dla każdego obrazu
    image_points : list
        Lista tablic współrzędnych 2D punktów na obrazie dla każdego obrazu
    image_size : tuple (width, height)
        Rozmiar obrazów w pikselach

    Zwraca:
    -------
    ret : float
        RMS błędu reprojekcji
    camera_matrix : ndarray, shape (3, 3)
        Macierz parametrów wewnętrznych kamery K
    dist_coeffs : ndarray, shape (5,)
        Współczynniki dystorsji [k1, k2, p1, p2, k3]
    rvecs : list
        Lista wektorów rotacji dla każdego obrazu
    tvecs : list
        Lista wektorów translacji dla każdego obrazu
    """
    # TODO: Zaimplementuj kalibrację kamery
    # Użyj cv2.calibrateCamera() z parametrami:
    # - object_points, image_points, image_size
    # - None dla początkowej macierzy kamery
    # - None dla początkowych współczynników dystorsji
    # - flags=0 (domyślne)

    pass

# Wykonanie kalibracji
print("Wykonywanie kalibracji kamery...")
print("=" * 60)

ret, camera_matrix, dist_coeffs, rvecs, tvecs = calibrate_camera(
    object_points_list,
    image_points_list,
    image_size
)

print(f"\n✓ Kalibracja zakończona pomyślnie!")
print(f"\nRMS błędu reprojekcji: {ret:.4f} pikseli")
print(f"\nMacierz kamery K:")
print(camera_matrix)
print(f"\nWspółczynniki dystorsji D:")
print(f"k1={dist_coeffs[0][0]:.6f}, k2={dist_coeffs[0][1]:.6f}, "
      f"p1={dist_coeffs[0][2]:.6f}, p2={dist_coeffs[0][3]:.6f}, k3={dist_coeffs[0][4]:.6f}")

# Ekstrakcja parametrów
fx = camera_matrix[0, 0]
fy = camera_matrix[1, 1]
cx = camera_matrix[0, 2]
cy = camera_matrix[1, 2]

print(f"\nParametry kamery:")
print(f"  Ogniskowa fx: {fx:.2f} pikseli")
print(f"  Ogniskowa fy: {fy:.2f} pikseli")
print(f"  Punkt główny cx: {cx:.2f} pikseli")
print(f"  Punkt główny cy: {cy:.2f} pikseli")
print(f"  Stosunek aspektów: {fx/fy:.4f}")


---
## **Kontynuacja ZADANIA 2**


In [ ]:
# ZADANIE 2.2: Analiza błędu reprojekcji

def calculate_reprojection_errors(object_points, image_points, rvecs, tvecs, 
                                   camera_matrix, dist_coeffs):
    """
    Oblicza błędy reprojekcji dla każdego obrazu i każdego punktu.

    Parametry:
    ----------
    object_points : list
        Lista tablic współrzędnych 3D punktów wzorca
    image_points : list
        Lista tablic współrzędnych 2D punktów na obrazie
    rvecs : list
        Lista wektorów rotacji
    tvecs : list
        Lista wektorów translacji
    camera_matrix : ndarray
        Macierz kamery K
    dist_coeffs : ndarray
        Współczynniki dystorsji

    Zwraca:
    -------
    mean_errors : ndarray
        Średni błąd reprojekcji dla każdego obrazu
    total_error : float
        Całkowity średni błąd reprojekcji
    all_errors : list
        Lista błędów dla każdego punktu na każdym obrazie
    """
    # TODO: Zaimplementuj obliczanie błędów reprojekcji
    # 1. Dla każdego obrazu użyj cv2.projectPoints() do reprojekcji punktów 3D
    # 2. Oblicz odległość euklidesową między punktami wykrytymi a reprojektowanymi
    #    Wzór: sqrt((x1-x2)^2 + (y1-y2)^2)
    # 3. Oblicz średni błąd dla każdego obrazu
    # 4. Oblicz całkowity średni błąd

    pass

# Obliczenie błędów
mean_errors, total_error, all_errors = calculate_reprojection_errors(
    object_points_list,
    image_points_list,
    rvecs,
    tvecs,
    camera_matrix,
    dist_coeffs
)

print("\nAnaliza błędu reprojekcji:")
print("=" * 60)
print(f"Całkowity średni błąd: {total_error:.4f} pikseli")
print(f"Minimalny błąd: {np.min(mean_errors):.4f} pikseli (obraz {np.argmin(mean_errors)+1})")
print(f"Maksymalny błąd: {np.max(mean_errors):.4f} pikseli (obraz {np.argmax(mean_errors)+1})")
print(f"Odchylenie standardowe: {np.std(mean_errors):.4f} pikseli")


---
# **ZADANIE 3: Korekcja dystorsji obrazu**

## **Cel zadania:**
Zastosowanie obliczonych parametrów kalibracji do usunięcia dystorsji z obrazów oraz porównanie obrazów przed i po korekcji.

## **Teoria:**

### **Proces korekcji dystorsji**
Korekcja dystorsji polega na odwróceniu zniekształceń wprowadzonych przez układ optyczny kamery:

1. **Undistortion**: Dla każdego piksela w obrazie wynikowym obliczamy, skąd powinien pochodzić piksel w obrazie zniekształconym
2. **Remapping**: Przepisujemy wartości pikseli z obrazu źródłowego do docelowego

### **Optymalna macierz kamery**
Parametr `alpha` kontroluje obszar widzenia:
- `alpha = 0`: Obraz wynikowy zawiera tylko prawidłowe piksele (przycięty)
- `alpha = 1`: Obraz wynikowy zawiera wszystkie piksele (z czarnymi obszarami)
- `alpha ∈ (0, 1)`: Kompromis między powyższymi

### **Funkcje OpenCV:**

#### `cv2.getOptimalNewCameraMatrix(cameraMatrix, distCoeffs, imageSize, alpha, newImgSize)`
Oblicza optymalną nową macierz kamery po korekcji dystorsji.
- **alpha**: Parametr kontrolujący obszar widzenia (0-1)
- **Zwraca**: (newCameraMatrix, validPixROI)
  - validPixROI: Region of Interest (x, y, w, h) z prawidłowymi pikselami

#### `cv2.undistort(src, cameraMatrix, distCoeffs, dst, newCameraMatrix)`
Usuwa dystorsję z obrazu.
- **src**: Obraz wejściowy
- **newCameraMatrix**: Zoptymalizowana macierz kamery
- **Zwraca**: Obraz po korekcji dystorsji

## **Implementacja:**


In [ ]:
# ZADANIE 3.1: Funkcja korekcji dystorsji

def undistort_image(image, camera_matrix, dist_coeffs, alpha=1.0):
    """
    Usuwa dystorsję z obrazu używając parametrów kalibracji.

    Parametry:
    ----------
    image : ndarray
        Obraz wejściowy do korekcji
    camera_matrix : ndarray
        Macierz kamery K
    dist_coeffs : ndarray
        Współczynniki dystorsji
    alpha : float, optional
        Parametr kontrolujący obszar widzenia (0-1), domyślnie 1.0

    Zwraca:
    -------
    undistorted : ndarray
        Obraz po korekcji dystorsji
    roi : tuple (x, y, w, h)
        Region of Interest - obszar prawidłowych pikseli
    new_camera_matrix : ndarray
        Zoptymalizowana macierz kamery
    """
    # TODO: Zaimplementuj korekcję dystorsji
    # 1. Pobierz rozmiar obrazu (h, w)
    # 2. Oblicz optymalną nową macierz kamery używając cv2.getOptimalNewCameraMatrix()
    #    Parametry: camera_matrix, dist_coeffs, (w, h), alpha, (w, h)
    # 3. Usuń dystorsję używając cv2.undistort()
    #    Parametry: image, camera_matrix, dist_coeffs, None, new_camera_matrix
    # 4. Zwróć obraz po korekcji, ROI i nową macierz kamery

    pass

# Test korekcji z różnymi wartościami alpha
test_image = calibration_images[valid_images[0]]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Oryginalny obraz
axes[0, 0].imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Oryginalny obraz (z dystorsją)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Korekcja z alpha=0
undist_alpha0, roi0, _ = undistort_image(test_image, camera_matrix, dist_coeffs, alpha=0.0)
x, y, w, h = roi0
undist_alpha0_cropped = undist_alpha0[y:y+h, x:x+w]
axes[0, 1].imshow(cv2.cvtColor(undist_alpha0_cropped, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('Korekcja z alpha=0 (przycięty)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

# Korekcja z alpha=0.5
undist_alpha05, roi05, _ = undistort_image(test_image, camera_matrix, dist_coeffs, alpha=0.5)
axes[1, 0].imshow(cv2.cvtColor(undist_alpha05, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Korekcja z alpha=0.5 (kompromis)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

# Korekcja z alpha=1
undist_alpha1, roi1, _ = undistort_image(test_image, camera_matrix, dist_coeffs, alpha=1.0)
axes[1, 1].imshow(cv2.cvtColor(undist_alpha1, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title('Korekcja z alpha=1 (pełny obraz)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


---
## **Kontynuacja ZADANIA 3**


In [ ]:
# ZADANIE 3.2: Porównanie obrazów przed i po korekcji z siatką

def visualize_distortion_correction(images, indices, camera_matrix, dist_coeffs, alpha=1.0):
    """
    Wizualizuje efekt korekcji dystorsji dla wybranych obrazów.

    Parametry:
    ----------
    images : list
        Lista wszystkich obrazów
    indices : list
        Indeksy obrazów do wizualizacji
    camera_matrix : ndarray
        Macierz kamery K
    dist_coeffs : ndarray
        Współczynniki dystorsji
    alpha : float
        Parametr kontrolujący obszar widzenia
    """
    # TODO: Zaimplementuj wizualizację porównawczą
    # Dla każdego wybranego obrazu:
    # 1. Wykonaj korekcję dystorsji
    # 2. Wyświetl oryginalny i skorygowany obraz obok siebie
    # 3. Dodaj siatkę pomocniczą (linie pionowe i poziome) do wizualizacji
    #    Użyj ax.axvline() i ax.axhline() z np.linspace()

    pass

# Wizualizacja dla wybranych obrazów
selected_indices = [valid_images[i] for i in [0, 3, 7, 11] if i < len(valid_images)]
visualize_distortion_correction(
    calibration_images, 
    selected_indices, 
    camera_matrix, 
    dist_coeffs, 
    alpha=1.0
)


---
# **ZADANIE 4: Wizualizacja układu współrzędnych 3D**

## **Cel zadania:**
Narysowanie osi układu współrzędnych 3D na obrazie kalibracyjnym w celu wizualizacji parametrów zewnętrznych kamery (rotacji i translacji).

## **Teoria:**

### **Układ współrzędnych kamery**
Parametry zewnętrzne (R, t) definiują transformację między:
- **Układem świata** (world coordinates): związany z wzorcem kalibracyjnym
- **Układem kamery** (camera coordinates): związany z kamerą

### **Reprezentacja Rodrigues**
Wektor rotacji `rvec` (3×1) to reprezentacja Rodrigues macierzy rotacji R (3×3):
- Kierunek wektora: oś rotacji
- Długość wektora: kąt rotacji w radianach

Konwersja:
```python
R, _ = cv2.Rodrigues(rvec)  # wektor → macierz
rvec, _ = cv2.Rodrigues(R)  # macierz → wektor
```

### **Projekcja osi 3D**
Punkty 3D reprezentujące osie:
- Początek: (0, 0, 0)
- Oś X: (length, 0, 0) - kolor czerwony
- Oś Y: (0, length, 0) - kolor zielony
- Oś Z: (0, 0, -length) - kolor niebieski

### **Funkcje OpenCV:**

#### `cv2.projectPoints(objectPoints, rvec, tvec, cameraMatrix, distCoeffs)`
Projektuje punkty 3D na płaszczyznę obrazu 2D.

#### `cv2.line(img, pt1, pt2, color, thickness)`
Rysuje linię na obrazie.

#### `cv2.putText(img, text, org, fontFace, fontScale, color, thickness)`
Dodaje tekst na obrazie.

## **Implementacja:**


In [ ]:
# ZADANIE 4.1: Funkcja rysująca osie 3D

def draw_3d_axes(image, camera_matrix, dist_coeffs, rvec, tvec, axis_length=50):
    """
    Rysuje osie układu współrzędnych 3D na obrazie.

    Parametry:
    ----------
    image : ndarray
        Obraz, na którym mają być narysowane osie
    camera_matrix : ndarray
        Macierz kamery K
    dist_coeffs : ndarray
        Współczynniki dystorsji
    rvec : ndarray
        Wektor rotacji (reprezentacja Rodrigues)
    tvec : ndarray
        Wektor translacji
    axis_length : float
        Długość osi w jednostkach rzeczywistych

    Zwraca:
    -------
    image_with_axes : ndarray
        Obraz z narysowanymi osiami
    """
    # TODO: Zaimplementuj rysowanie osi 3D
    # 1. Zdefiniuj punkty 3D dla początku i końców osi X, Y, Z
    #    axis_points = np.float32([[0,0,0], [axis_length,0,0], [0,axis_length,0], [0,0,-axis_length]])
    # 2. Projektuj punkty 3D na płaszczyznę obrazu używając cv2.projectPoints()
    # 3. Narysuj linie reprezentujące osie:
    #    - Oś X: czerwona (0, 0, 255) w BGR
    #    - Oś Y: zielona (0, 255, 0) w BGR
    #    - Oś Z: niebieska (255, 0, 0) w BGR
    #    Użyj cv2.line() z grubością 3
    # 4. Dodaj etykiety 'X', 'Y', 'Z' używając cv2.putText()

    pass

# Wizualizacja osi 3D na wybranych obrazach
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.ravel()

for i in range(min(9, len(valid_images))):
    img_idx = valid_images[i]
    img = calibration_images[img_idx].copy()

    img_with_axes = draw_3d_axes(
        img,
        camera_matrix,
        dist_coeffs,
        rvecs[i],
        tvecs[i],
        axis_length=75
    )

    axes[i].imshow(cv2.cvtColor(img_with_axes, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Obraz {img_idx+1} - Układ współrzędnych 3D', fontweight='bold')
    axes[i].axis('off')

for i in range(min(9, len(valid_images)), 9):
    axes[i].axis('off')

plt.tight_layout()
plt.show()


---
## **Kontynuacja ZADANIA 4**


In [ ]:
# ZADANIE 4.2: Analiza parametrów zewnętrznych

def analyze_extrinsic_parameters(rvecs, tvecs):
    """
    Analizuje parametry zewnętrzne (rotacje i translacje) dla wszystkich obrazów.

    Parametry:
    ----------
    rvecs : list
        Lista wektorów rotacji
    tvecs : list
        Lista wektorów translacji

    Zwraca:
    -------
    rotation_angles : ndarray
        Kąty rotacji w stopniach dla każdego obrazu
    distances : ndarray
        Odległości kamery od wzorca dla każdego obrazu
    rotation_matrices : list
        Lista macierzy rotacji dla każdego obrazu
    """
    # TODO: Zaimplementuj analizę parametrów zewnętrznych
    # 1. Dla każdego rvec:
    #    - Oblicz kąt rotacji: np.linalg.norm(rvec) * 180 / np.pi (stopnie)
    #    - Konwertuj do macierzy rotacji używając cv2.Rodrigues()
    # 2. Dla każdego tvec:
    #    - Oblicz odległość: np.linalg.norm(tvec)
    # 3. Zwróć tablice kątów, odległości i listę macierzy rotacji

    pass

# Analiza parametrów
rotation_angles, distances, rotation_matrices = analyze_extrinsic_parameters(rvecs, tvecs)

print("\nAnaliza parametrów zewnętrznych:")
print("=" * 60)
print(f"Średni kąt rotacji: {np.mean(rotation_angles):.2f}°")
print(f"Zakres kątów rotacji: {np.min(rotation_angles):.2f}° - {np.max(rotation_angles):.2f}°")
print(f"\nŚrednia odległość kamery od wzorca: {np.mean(distances):.2f} mm")
print(f"Zakres odległości: {np.min(distances):.2f} - {np.max(distances):.2f} mm")


---
# **ZADANIE 5: Zapisanie i wczytanie parametrów kalibracji**

## **Cel zadania:**
Zapisanie obliczonych parametrów kalibracji do pliku w celu późniejszego wykorzystania oraz wczytanie i weryfikacja zapisanych parametrów.

## **Teoria:**

### **Format zapisu**
Parametry kalibracji zapisujemy w formacie XML lub YAML używając `cv2.FileStorage`.

### **Parametry do zapisania:**
- Macierz kamery K
- Współczynniki dystorsji D
- Rozmiar obrazu
- RMS błędu reprojekcji
- Data kalibracji

### **Funkcje OpenCV:**

#### `cv2.FileStorage(filename, flags)`
Otwiera plik do zapisu/odczytu parametrów.
- **flags**: `cv2.FILE_STORAGE_WRITE` lub `cv2.FILE_STORAGE_READ`

## **Implementacja:**


In [ ]:
# ZADANIE 5.1: Zapisanie parametrów kalibracji

def save_calibration_parameters(filename, camera_matrix, dist_coeffs, image_size, rms_error):
    """
    Zapisuje parametry kalibracji do pliku XML.

    Parametry:
    ----------
    filename : str
        Nazwa pliku do zapisu
    camera_matrix : ndarray
        Macierz kamery K
    dist_coeffs : ndarray
        Współczynniki dystorsji
    image_size : tuple
        Rozmiar obrazów (width, height)
    rms_error : float
        RMS błędu reprojekcji
    """
    # TODO: Zaimplementuj zapisanie parametrów
    # Użyj cv2.FileStorage do zapisu w formacie XML
    # Zapisz: camera_matrix, dist_coeffs, image_width, image_height, rms_error

    pass

# Zapisanie parametrów
save_calibration_parameters(
    'camera_calibration.xml',
    camera_matrix,
    dist_coeffs,
    image_size,
    ret
)

print("✓ Parametry kalibracji zapisane do pliku: camera_calibration.xml")


---
## **Kontynuacja ZADANIA 5**


In [ ]:
# ZADANIE 5.2: Wczytanie parametrów kalibracji

def load_calibration_parameters(filename):
    """
    Wczytuje parametry kalibracji z pliku XML.

    Parametry:
    ----------
    filename : str
        Nazwa pliku do odczytu

    Zwraca:
    -------
    camera_matrix : ndarray
        Macierz kamery K
    dist_coeffs : ndarray
        Współczynniki dystorsji
    image_size : tuple
        Rozmiar obrazów (width, height)
    rms_error : float
        RMS błędu reprojekcji
    """
    # TODO: Zaimplementuj wczytanie parametrów
    # Użyj cv2.FileStorage do odczytu z pliku XML
    # Wczytaj wszystkie zapisane parametry

    pass

# Wczytanie i weryfikacja
loaded_camera_matrix, loaded_dist_coeffs, loaded_image_size, loaded_rms = load_calibration_parameters(
    'camera_calibration.xml'
)

print("\n✓ Parametry kalibracji wczytane z pliku")
print("\nWeryfikacja wczytanych parametrów:")
print(f"Macierz kamery:\n{loaded_camera_matrix}")
print(f"\nWspółczynniki dystorsji:\n{loaded_dist_coeffs}")
print(f"\nRozmiar obrazu: {loaded_image_size}")
print(f"RMS błędu: {loaded_rms:.4f}")


---
# **PODSUMOWANIE**

## **Wykonane zadania:**

### ✓ **ZADANIE 1: Przygotowanie danych kalibracyjnych**
- Generowanie współrzędnych 3D wzorca szachownicy
- Wykrywanie narożników z subpikselową dokładnością (`cv2.findChessboardCorners`, `cv2.cornerSubPix`)
- Przetworzenie wszystkich obrazów kalibracyjnych

### ✓ **ZADANIE 2: Kalibracja kamery**
- Obliczenie macierzy kamery K i współczynników dystorsji D (`cv2.calibrateCamera`)
- Analiza błędu reprojekcji (`cv2.projectPoints`)
- Wizualizacja rozkładu błędów

### ✓ **ZADANIE 3: Korekcja dystorsji**
- Usunięcie dystorsji z obrazów (`cv2.getOptimalNewCameraMatrix`, `cv2.undistort`)
- Porównanie obrazów przed i po korekcji
- Wpływ parametru alpha na wynik korekcji

### ✓ **ZADANIE 4: Wizualizacja układu współrzędnych 3D**
- Rysowanie osi układu współrzędnych na obrazach (`cv2.projectPoints`, `cv2.line`)
- Analiza parametrów zewnętrznych (rotacja i translacja)
- Konwersja reprezentacji Rodrigues (`cv2.Rodrigues`)

### ✓ **ZADANIE 5: Zapisanie parametrów kalibracji**
- Zapis parametrów do pliku XML (`cv2.FileStorage`)
- Wczytanie i weryfikacja parametrów

---

## **Kluczowe funkcje OpenCV:**

| Funkcja | Zastosowanie |
|---------|--------------|
| `cv2.findChessboardCorners()` | Wykrywanie narożników szachownicy |
| `cv2.cornerSubPix()` | Rafinacja pozycji narożników (subpikselowa dokładność) |
| `cv2.drawChessboardCorners()` | Wizualizacja wykrytych narożników |
| `cv2.calibrateCamera()` | Kalibracja kamery (obliczenie K i D) |
| `cv2.projectPoints()` | Projekcja punktów 3D na płaszczyznę obrazu 2D |
| `cv2.getOptimalNewCameraMatrix()` | Optymalizacja macierzy kamery po korekcji |
| `cv2.undistort()` | Usunięcie dystorsji z obrazu |
| `cv2.Rodrigues()` | Konwersja reprezentacji rotacji (wektor ↔ macierz) |
| `cv2.FileStorage()` | Zapis/odczyt parametrów kalibracji |

---

## **Parametry kamery:**

### **Macierz kamery K:**
$$
K = \begin{bmatrix}
f_x & 0 & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$

- **fx, fy**: Ogniskowa w pikselach (oś X i Y)
- **cx, cy**: Punkt główny (optical center)

### **Współczynniki dystorsji D:**
$$
D = [k_1, k_2, p_1, p_2, k_3]
$$

- **k1, k2, k3**: Dystorsja radialna
- **p1, p2**: Dystorsja tangencjalna

---

## **Zastosowania kalibracji kamery:**

1. **Fotogrametria i rekonstrukcja 3D**
   - Pomiary odległości i wymiarów obiektów
   - Tworzenie modeli 3D ze zdjęć

2. **Rozszerzona rzeczywistość (AR)**
   - Precyzyjne nakładanie obiektów wirtualnych na rzeczywiste

3. **Robotyka i nawigacja**
   - Lokalizacja i mapowanie (SLAM)
   - Sterowanie robotami mobilnymi

4. **Systemy wizyjne w pojazdach autonomicznych**
   - Detekcja przeszkód i pasów ruchu
   - Estymacja odległości

5. **Pomiary przemysłowe**
   - Kontrola jakości
   - Inspekcja wizualna

---

## **Wskazówki praktyczne:**

### **Jak uzyskać dobrą kalibrację:**

1. **Jakość obrazów kalibracyjnych:**
   - Używaj ostrych, dobrze oświetlonych zdjęć
   - Unikaj rozmycia ruchu
   - Szachownica powinna zajmować znaczną część obrazu

2. **Różnorodność pozycji:**
   - Fotografuj szachownicę z różnych kątów
   - Umieszczaj wzorzec w różnych częściach obrazu
   - Zmieniaj odległość od kamery

3. **Liczba obrazów:**
   - Minimum 10-15 obrazów
   - Optymalne: 20-30 obrazów
   - Więcej obrazów = lepsza kalibracja

4. **Ocena jakości kalibracji:**
   - RMS błędu reprojekcji < 0.5 piksela: doskonała
   - RMS błędu reprojekcji < 1.0 piksela: dobra
   - RMS błędu reprojekcji > 1.0 piksela: wymaga poprawy

---

**Gratulacje! Ukończyłeś laboratorium z kalibracji kamery.**

Teraz możesz wykorzystać obliczone parametry do:
- Korekcji dystorsji w aplikacjach wizyjnych
- Rekonstrukcji 3D
- Aplikacji AR/VR
- Pomiarów metrycznych na obrazach
